# 第3回：画像認識の結果をデータとして分析する

**このNotebookでできるようになること**

1. 映像から検出した「人数」を時刻とともに記録し，表（DataFrame）にする
2. 時系列グラフとして可視化する
3. 移動平均でノイズを減らし，変化点検出で「状況が変わった時刻」を見つける
4. CSVに保存して，後から再利用する

画像認識アプリの価値は「検出できる」ことではなく，**検出結果を集計・分析して意味を取り出す**ことにあります．

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def set_japanese_font():
    # グラフの日本語が □ にならないように，OSにある日本語フォントを設定する
    from matplotlib import font_manager as fm
    available = {f.name for f in fm.fontManager.ttflist}
    for name in ["Meiryo", "Yu Gothic", "MS Gothic", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
        if name in available:
            plt.rcParams["font.family"] = name
            return name
    return None

print("font:", set_japanese_font())

## 1. 人通りの変化を記録する

動画 `vtest.avi` に対して10フレームごとに人物検出を行い，**時刻（秒）と人数** を記録します．

In [ ]:
cap = cv2.VideoCapture("data/vtest.avi")
fps = cap.get(cv2.CAP_PROP_FPS)

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
hogParams = {"winStride": (4, 4), "padding": (8, 8), "scale": 1.05, "hitThreshold": 0}

records = []                                   # [{"time": 秒, "people": 人数}, ...]
num = 0
print("分析を開始します（動画の長さによって数分かかります）")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if num % 10 == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        human, _ = hog.detectMultiScale(gray, **hogParams)
        records.append({"time": num / fps, "people": len(human)})
    num += 1
cap.release()

df = pd.DataFrame(records)
print("分析を終了しました．記録数:", len(df))
df.head()

## 2. グラフにする

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["time"], df["people"])
plt.xlabel("時間（秒）")
plt.ylabel("検出人数")
plt.ylim(0, 8)
plt.title("時刻ごとの検出人数")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 基本統計量：平均・最大・最小など
df["people"].describe()

## 3. 移動平均でノイズを減らす

検出器は毎フレーム同じ精度では働かないので，人数はギザギザします．**移動平均**（前後数点の平均）で滑らかにすると傾向が見やすくなります．

In [ ]:
WINDOW = 5                                     # ← 平均する点数（大きいほど滑らか）
df["people_ma"] = df["people"].rolling(WINDOW, center=True).mean()

plt.figure(figsize=(10, 4))
plt.plot(df["time"], df["people"], alpha=0.4, label="生データ")
plt.plot(df["time"], df["people_ma"], linewidth=2, label=f"移動平均（{WINDOW}点）")
plt.xlabel("時間（秒）")
plt.ylabel("検出人数")
plt.ylim(0, 8)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. 変化点検出（PELTアルゴリズム）

「人数が急に増えた／減った時刻」を自動で見つけます．`ruptures` の PELT は，データを区間に分けたときの当てはまりの良さと区間数のバランス（ペナルティ）で変化点を決めます．

In [ ]:
import ruptures as rpt

signal = df["people"].values
PENALTY = 3                                    # ← 大きいほど変化点が少なくなる

algo = rpt.Pelt(model="l1").fit(signal)
change_points = algo.predict(pen=PENALTY)
print("検出された変化点（インデックス）:", change_points)

plt.figure(figsize=(10, 4))
plt.plot(df["time"], signal, label="検出人数")
for cp in change_points[:-1]:                  # 最後の要素はデータの終端
    plt.axvline(df["time"].iloc[cp], color="red", linestyle="--")
plt.xlabel("時間（秒）")
plt.ylabel("検出人数")
plt.ylim(0, 8)
plt.legend()
plt.title(f"変化点検出（PELT, penalty={PENALTY}）")
plt.show()

## 5. CSVに保存する・読み込む

検出には時間がかかるので，結果は必ずCSVに保存しておきます．分析や可視化は保存したCSVから何度でもやり直せます．

In [ ]:
df.to_csv("data/people_count.csv", index=False, encoding="utf-8-sig")
print("data/people_count.csv に保存しました")

df2 = pd.read_csv("data/people_count.csv")
df2.head()

## 6. 区間ごとの集計

「何秒から何秒までは平均何人だったか」のように，変化点で区切った区間ごとに集計します．発表ではこうした **数値の要約** が説得力を持ちます．

In [ ]:
bounds = [0] + change_points
rows = []
for s, e in zip(bounds[:-1], bounds[1:]):
    seg = df.iloc[s:e]
    rows.append({"開始(秒)": round(seg["time"].iloc[0], 1), "終了(秒)": round(seg["time"].iloc[-1], 1),
                 "平均人数": round(seg["people"].mean(), 2), "最大人数": int(seg["people"].max())})
pd.DataFrame(rows)

## 7. Webカメラで人数をリアルタイム記録する（.py スクリプトとしても用意）

同じ処理を Webカメラに対して行うと，「教室の在室人数の推移」「店の前の通行量」のような **自分たちのデータ** を作れます．
`scripts/count_people_webcam.py` を Anaconda Prompt から実行すると，`q` で終了するまでの人数を CSV に記録します．

```bash
python scripts/count_people_webcam.py --out data/webcam_people.csv
```

In [ ]:
# [SKIP-TEST]
import time

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

records = []
t0 = time.time()
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    human, _ = hog.detectMultiScale(gray, winStride=(8, 8), padding=(32, 32), scale=1.05)
    for (x, y, w, h) in human:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 2)
    records.append({"time": round(time.time() - t0, 2), "people": len(human)})
    cv2.putText(frame, f"people: {len(human)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cap.release()
cv2.destroyAllWindows()

pd.DataFrame(records).to_csv("data/webcam_people.csv", index=False)
print("data/webcam_people.csv に", len(records), "件保存しました")

## まとめ

| できること | 使ったもの |
|---|---|
| 検出結果を表にする | `pandas.DataFrame` |
| 時系列グラフ | `matplotlib` |
| ノイズ除去 | 移動平均 `rolling().mean()` |
| 状況の変化を見つける | 変化点検出 `ruptures.Pelt` |
| 結果の保存・再利用 | `to_csv`, `read_csv` |

**次回（第4回）**：深層学習（MediaPipe の姿勢推定，YOLOv5 の物体検出）を使って，検出できる対象を「人」から「80種類の物体」「身体の関節」へ広げます．